# Steam Exploratory Data Analysis and Recommender System Design

Inspect four local Steam files and assess which signals support a personalized game recommender. The main experiment ranks games within a user's existing library.

Run this notebook from the workspace root or the `steam` directory with Python, pandas, and Jupyter. The game catalog, Australian reviews, and game libraries are analyzed in full. The largest review file is sampled across multiple locations.

In [ ]:
from pathlib import Path
from ast import literal_eval
from collections import Counter
from itertools import islice
import re

import pandas as pd
from IPython.display import display

root = (Path('steam') if Path('steam').is_dir() else Path('.')) / 'data'
names = ['steam_games.json', 'australian_user_reviews.json',
         'australian_users_items.json', 'steam_reviews.json']
paths = {name: root / name for name in names}

def records(name):
    with paths[name].open(encoding='utf-8-sig') as source:
        for line_number, line in enumerate(source, 1):
            if line.strip():
                try:
                    yield literal_eval(line)
                except (ValueError, SyntaxError) as error:
                    raise ValueError(f'{name}, line {line_number}') from error

display(pd.Series({name: path.stat().st_size / 1e6 for name, path in paths.items()},
                  name='size (MB)').round(1).to_frame())
for name in names:
    row = next(records(name))
    preview = {key: (value[:1] if isinstance(value, list) else value)
               for key, value in row.items()}
    print(name, '\n', repr(preview)[:1400], '\n')

steam_games.json 
 {'publisher': 'Kotoshiro', 'genres': ['Action'], 'app_name': 'Lost Summoner Kitty', 'title': 'Lost Summoner Kitty', 'url': 'http://store.steampowered.com/app/761140/Lost_Summoner_Kitty/', 'release_date': '2018-01-04', 'tags': ['Strategy'], 'discount_price': 4.49, 'reviews_url': 'http://steamcommunity.com/app/761140/reviews/?browsefilter=mostrecent&p=1', 'specs': ['Single-player'], 'price': 4.99, 'early_access': False, 'id': '761140', 'developer': 'Kotoshiro'} 

australian_user_reviews.json 
 {'user_id': '76561197970982479', 'user_url': 'http://steamcommunity.com/profiles/76561197970982479', 'reviews': [{'funny': '', 'posted': 'Posted November 5, 2011.', 'last_edited': '', 'item_id': '1250', 'helpful': 'No ratings yet', 'recommend': True, 'review': 'Simple yet with great replayability. In my opinion does "zombie" hordes and team work better than left 4 dead plus has a global leveling system. Alot of down to earth "zombie" splattering fun for the whole family. Amazed t

,size (MB)
steam_games.json,21.2
australian_user_reviews.json,26.3
australian_users_items.json,553.1
steam_new.json,4273.2


## Game Catalog and Australian Reviews

Flatten each user's nested reviews into one row per review. Check duplicates, catalog coverage, class imbalance, and posting dates. Dates without a year are treated as missing.

In [ ]:
games = pd.DataFrame(records(names[0]))
reviews = pd.DataFrame(
    {'user_id': user['user_id'], **review}
    for user in records(names[1]) for review in user['reviews']
)
game_ids = set(games['id'].dropna())
review_users = set(reviews['user_id'])
review_pairs = set(zip(reviews['user_id'], reviews['item_id']))

display(games[['id', 'app_name', 'genres', 'price', 'release_date']].head())
display(reviews[['item_id', 'posted', 'recommend', 'helpful', 'review']].head())
display(pd.DataFrame({
    'missing %': games.isna().mean().mul(100).round(1),
    'nonmissing': games.notna().sum()
}))
display(games['genres'].explode().value_counts().head(10).rename('games').to_frame())

raw_review_rows = len(reviews)
reviews = reviews.drop_duplicates().assign(
    date=lambda df: pd.to_datetime(
        df['posted'].str.extract(r'^Posted (.*?, \d{4})\.$', expand=False),
        format='%B %d, %Y', errors='coerce'
    )
)
display(pd.Series({
    'catalog rows': len(games), 'unique catalog IDs': len(game_ids),
    'raw review rows': raw_review_rows, 'exact distinct reviews': len(reviews),
    'review users': len(review_users),
    'reviewed games': reviews['item_id'].nunique(),
    'review user-game pairs': len(review_pairs),
    'review rows with catalog metadata': reviews['item_id'].isin(game_ids).mean(),
    'reviews with parsed posting dates': reviews['date'].notna().mean(),
    'earliest parsed date': reviews['date'].min(),
    'latest parsed date': reviews['date'].max()
}).to_frame('value'))

,id,app_name,genres,price,release_date
0,761140,Lost Summoner Kitty,"[Action, Casual, Indie, Simulation, Strategy]",4.99,2018-01-04
1,643980,Ironbound,"[Free to Play, Indie, RPG, Strategy]",Free To Play,2018-01-04
2,670290,Real Pool 3D - Poolians,"[Casual, Free to Play, Indie, Simulation, Sports]",Free to Play,2017-07-24
3,767400,弹炸人2222,"[Action, Adventure, Casual]",0.99,2017-12-07
4,773570,Log Challenge,NaN,2.99,NaN


,item_id,posted,recommend,helpful,review
0,1250,"Posted November 5, 2011.",True,No ratings yet,Simple yet with great replayability. In my opi...
1,22200,"Posted July 15, 2011.",True,No ratings yet,It's unique and worth a playthrough.
2,43110,"Posted April 21, 2011.",True,No ratings yet,Great atmosphere. The gunplay can be a bit chu...
3,251610,"Posted June 24, 2014.",True,15 of 20 people (75%) found this review helpful,I know what you think when you see this title ...
4,227300,"Posted September 8, 2013.",True,0 of 1 people (0%) found this review helpful,For a simple (it's actually not all that simpl...


,missing %,nonmissing
publisher,25.1,24083
genres,10.2,28852
app_name,0.0,32133
title,6.4,30085
url,0.0,32135
release_date,6.4,30068
tags,0.5,31972
discount_price,99.3,225
reviews_url,0.0,32133
specs,2.1,31465


,games
genres,
Indie,15858
Action,11321
Casual,8282
Adventure,8243
Strategy,6957
Simulation,6699
RPG,5479
Free to Play,2031
Early Access,1462


,value
catalog rows,32135
unique catalog IDs,32132
raw review rows,59305
exact distinct reviews,58431
review users,25458
reviewed games,3682
review user-game pairs,58431
review rows with catalog metadata,0.910424
reviews with parsed posting dates,0.830005
earliest parsed date,2010-10-16 00:00:00


In [ ]:
def bars(counts, label):
    table = counts.rename(label).to_frame()
    display(table.style.format({label: '{:,}'}).bar(subset=[label], color='#9bc7df'))

bars(reviews['recommend'].value_counts(dropna=False), 'reviews')
bars(reviews['date'].dt.year.dropna().astype(int).value_counts().sort_index(), 'reviews')
votes = reviews['helpful'].str.extract(r'([\d,]+) of ([\d,]+)')
votes = votes.replace(',', '', regex=True).astype('Int64')
reviews['helpful_votes'] = votes[0]
reviews['total_votes'] = votes[1]
display(pd.Series({
    'recommend share': reviews['recommend'].mean(),
    'reviews with at least 5 votes': reviews['total_votes'].ge(5).sum(),
    'median review words': reviews['review'].str.split().str.len().median()
}).to_frame('value'))

,reviews
recommend,
True,"51,692"
False,"6,739"


,reviews
date,
2010,66
2011,530
2012,"1,201"
2013,"6,713"
2014,"21,834"
2015,"18,154"


,value
recommend share,0.884667
reviews with at least 5 votes,7296.000000
median review words,14.000000


## Australian Game Libraries and Playtime

Read one user's library at a time and aggregate statistics. Interaction counts refer to raw library entries. The table also reports the number of distinct users. Playtime is measured in minutes, with zero-minute entries shown separately.

In [ ]:
library_users, library_games = set(), set()
matched_review_pairs = set()
played_by_game = Counter()
library_sizes = Counter()
duration_bands = Counter()
library_rows = item_rows = catalog_item_rows = two_week_positive = 0

duration_limits = [
    (-1, 'negative'), (0, '0'), (60, '1–60'), (600, '61–600'),
    (6000, '601–6,000'), (float('inf'), '6,001+')
]

for user in records(names[2]):
    user_id = user['user_id']
    library_rows += 1
    library_users.add(user_id)
    library_sizes[len(user['items'])] += 1
    for item in user['items']:
        game_id = item['item_id']
        minutes = item['playtime_forever']
        library_games.add(game_id)
        item_rows += 1
        catalog_item_rows += game_id in game_ids
        two_week_positive += item['playtime_2weeks'] > 0
        pair = (user_id, game_id)
        if pair in review_pairs:
            matched_review_pairs.add(pair)
        if minutes > 0:
            played_by_game[game_id] += 1
        band = next(label for limit, label in duration_limits if minutes <= limit)
        duration_bands[band] += 1

display(pd.Series({
    'library user rows': library_rows,
    'distinct library users': len(library_users),
    'library game IDs': len(library_games),
    'library item rows': item_rows,
    'library item rows with catalog metadata': catalog_item_rows / item_rows,
    'review users found in library': len(review_users & library_users) / len(review_users),
    'review pairs found in library': len(matched_review_pairs) / len(review_pairs),
    'positive 2-week playtime share': two_week_positive / item_rows,
    'median library size (items)': pd.Series(library_sizes).sort_index().cumsum().ge(library_rows / 2).idxmax()
}).to_frame('value'))
bars(pd.Series(duration_bands).reindex([label for _, label in duration_limits]).fillna(0).astype(int), 'items')
display(pd.DataFrame(played_by_game.most_common(10), columns=['id', 'played rows']).merge(
    games[['id', 'app_name']].drop_duplicates('id'), on='id', how='left'
))

,value
library user rows,8.831000e+04
distinct library users,8.762600e+04
library game IDs,1.097800e+04
library item rows,5.153209e+06
library item rows with catalog metadata,8.333171e-01
review users found in library,9.998429e-01
review pairs found in library,7.562082e-01
positive 2-week playtime share,2.687685e-02
median library size (items),2.600000e+01


,items
negative,0
0,"1,867,963"
1–60,"964,278"
61–600,"1,337,052"
"601–6,000","818,933"
"6,001+","164,983"


,id,played rows,app_name
0,730,43279,Counter-Strike: Global Offensive
1,4000,42074,Garry's Mod
2,550,34899,Left 4 Dead 2
3,304930,34465,Unturned
4,105600,28541,Terraria
5,620,22696,Portal 2
6,218620,22378,PAYDAY 2
7,240,21671,Counter-Strike: Source
8,72850,21642,NaN
9,230410,21087,Warframe


## Large Review File

Sample up to 500 complete records at each of 20 evenly spaced byte offsets in `steam_new.json`. Skip the partial line at nonzero offsets. Reviews may be clustered by game in the file, so the statistics describe the sampled records.

In [ ]:
def sample_new_reviews(path, blocks=20, rows_per_block=500):
    with path.open('rb') as source:
        size = path.stat().st_size
        for block in range(blocks):
            offset = size * block // blocks
            source.seek(offset)
            if offset:
                source.readline()
            for line in islice(source, rows_per_block):
                if line.strip():
                    yield {'sample_block': block, **literal_eval(line.decode('utf-8-sig'))}

new = pd.DataFrame(sample_new_reviews(paths[names[3]]))
new_dates = pd.to_datetime(new['date'], format='%Y-%m-%d', errors='coerce')
display(new.head())
display(new.isna().sum().sort_values(ascending=False).rename('missing in sample').to_frame())
display(pd.Series({
    'sample rows': len(new),
    'sample product IDs': new['product_id'].nunique(),
    'sample rows with catalog metadata': new['product_id'].isin(game_ids).mean(),
    'sample rows with usernames matching Australian review user_id strings':
        new['username'].isin(review_users).mean(),
    'earliest parsed sample date': new_dates.min(),
    'latest parsed sample date': new_dates.max()
}).to_frame('value'))
display(pd.to_numeric(new['hours'], errors='coerce').describe(percentiles=[.5, .9, .99]).to_frame())
new_ids = new['user_id'].dropna().astype(str)
display(pd.Series({
    'sample rows with explicit user IDs': len(new_ids),
    'nonmissing ID rows matching Australian review user_id strings': new_ids.isin(review_users).sum(),
}).to_frame('value'))


,sample_block,username,hours,products,product_id,page_order,date,text,early_access,page,found_funny,compensation,user_id
0,0,Chaos Syren,0.1,41.0,725280,0,2017-12-17,This would not be acceptable as an entertainme...,False,1,NaN,NaN,NaN
1,0,₮ʜᴇ Wᴀʀᴛᴏɴ,51.1,769.0,328100,0,2017-12-27,looks like a facebook game,False,1,NaN,NaN,NaN
2,0,hello?<,14.6,2.0,328100,1,2017-10-16,Better than Minecraft,False,1,2.0,Product received for free,NaN
3,0,Cyderine916,5.0,64.0,35140,0,2018-01-04,I love and idolized Batman and this game is Ma...,False,1,NaN,NaN,NaN
4,0,DarklyThinking,16.6,577.0,35140,1,2018-01-04,Still worth playing in 2018.\nProbably my favo...,False,1,NaN,NaN,76561198007483075


,missing in sample
compensation,9806
found_funny,8358
user_id,5951
hours,55
products,24
product_id,0
username,0
sample_block,0
page_order,0
early_access,0


,value
sample rows,10000
sample product IDs,362
sample rows with catalog metadata,1.0
sample rows with usernames matching Australian review user_id strings,0.0006
earliest parsed sample date,2010-12-23 00:00:00
latest parsed sample date,2018-01-05 00:00:00


,hours
count,9945.000000
mean,117.607139
std,377.901308
min,0.000000
50%,15.400000
90%,239.480000
99%,1820.368000
max,8731.600000


,value
sample rows with explicit user IDs,4049
nonmissing ID rows matching Australian review user_id strings,18
